# Generative AI & LLM Engineering: From Concepts to Practice
**ADSP 32027 · Week 1 hands-on (after class)**

| Part | Topics | Status |
|------|--------|--------|
| **Part 1 — LLM Fundamentals** | Open- vs. closed-book QA, in-context learning (fine-tuning theory optional) | Required |
| **Part 2 — Prompt Engineering** | System/user roles, zero-/few-shot, role and chain prompting | Required |
| **Part 3 — Text Splitters** | Character, recursive, NLTK, Markdown, token-based splitting | Optional preview of Week 2 (RAG chunking) |
| **Part 4 — LangChain LCEL** | LCEL chains, conversation memory, composing chains | Optional preview of Week 3 (LangChain) |

- **Runtime:** Colab CPU is fine (no GPU needed).
- **API key:** Parts 2–4 call the OpenAI API. Add `OPENAI_API_KEY` under Colab Secrets (key icon in the left sidebar) before running the setup cell.
- **Model:** set once in the setup cell (`MODEL`). Change it there if needed.

## Setup

Run the install cell once, then **restart the runtime** before proceeding.

In [ ]:
# Install all dependencies (Colab-compatible)
!pip install -q \
    langchain langchain-openai langchain-community langchain-text-splitters \
    openai tiktoken pypdf nltk \
    transformers torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
import os

# Try Colab Secrets first (Colab → left sidebar → key icon → add OPENAI_API_KEY)
try:
    from google.colab import userdata
    _key = userdata.get("OPENAI_API_KEY")
    if _key:
        os.environ["OPENAI_API_KEY"] = _key
except Exception:
    pass

# Fall back to interactive prompt if not already set
if not os.environ.get("OPENAI_API_KEY"):
    import getpass
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

MODEL = "gpt-4.1-mini"  # used by every API call below; change it here only

print("OpenAI API key loaded. Model:", MODEL)

OpenAI API key loaded. Model: gpt-4.1-mini


---
# Part 1 — LLM Fundamentals

1. Open-book vs. closed-book question answering
2. In-context learning: zero-shot, one-shot, few-shot
3. Fine-tuning and instruction fine-tuning (theory)

Part 1 uses small open models that run on CPU (`facebook/opt-125m`, ~250 MB, for text generation). No API key needed for this part.

## 1.1 QA Models — Open-Book vs. Closed-Book

| Type | How it answers | When to use |
|------|---------------|-------------|
| **Open-Book QA** | Reads a supplied *context* document to extract the answer | When the answer lives in a specific document you provide |
| **Closed-Book QA** | Relies entirely on knowledge encoded during training | When you need quick answers from general world knowledge |

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

# NOTE: transformers v5.3 removed the "question-answering" pipeline task string entirely.
# We load the model and tokenizer directly — this also exposes the span-extraction
# mechanics (start/end logits) that the pipeline hides.
#
# One-liner alternative that still uses pipeline() (auto-detects task from model card):
#   from transformers import pipeline
#   qa = pipeline(model="deepset/roberta-base-squad2")
#   result = qa(question=question, context=context); print(result["answer"])

model_name = "deepset/roberta-base-squad2"
qa_tokenizer = AutoTokenizer.from_pretrained(model_name)
qa_model = AutoModelForQuestionAnswering.from_pretrained(model_name)

context = """
The theory of relativity, developed by Albert Einstein, revolutionized our
understanding of space, time, and gravity. It introduced concepts such as
spacetime and the equivalence of mass and energy (E = mc²).
"""
question = "Who developed the theory of relativity?"

# Tokenize question + context together (model sees them as a single sequence)
inputs = qa_tokenizer(question, context, return_tensors="pt")

with torch.no_grad():
    outputs = qa_model(**inputs)

# The model outputs two logit vectors — one per token for start, one for end.
# argmax gives the most likely start/end token positions of the answer span.
start_idx = outputs.start_logits.argmax()
end_idx   = outputs.end_logits.argmax() + 1  # +1 because slice is exclusive

answer = qa_tokenizer.decode(inputs["input_ids"][0][start_idx:end_idx], skip_special_tokens=True)
print("Open-Book answer:", answer)
# Expected: Albert Einstein

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  496MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Open-Book answer:  Albert Einstein


In [ ]:
from transformers import pipeline

# Small model for the closed-book demo — point is to show the MECHANISM, not output quality.
pipeline_model = pipeline(
    "text-generation",
    model="facebook/opt-125m",
    max_new_tokens=50,
    pad_token_id=2,
)

question = "Who developed the theory of relativity?"
result = pipeline_model(question)
print("Closed-Book answer (opt-125m):", result[0]["generated_text"])

config.json:   0%|          | 0.00/651 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  251MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  251MB            

...

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=50) and `max_length`(=21) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for

Closed-Book answer (opt-125m): Who developed the theory of relativity?
I think it might be called Einstein’s Theory of Relativity.
I think it might be called Einstein’s Theory of Relativity.


### Comparison Summary

| | Open-Book QA | Closed-Book QA |
|---|---|---|
| **Speed** | Slower (retrieval step) | Faster |
| **Freshness** | Can use up-to-date documents | Limited to training cutoff |
| **Accuracy** | High for in-context facts | Depends on training data |
| **Complexity** | Requires document retrieval pipeline | Simpler architecture |

> **Key insight:** Modern RAG systems combine both: a retriever fetches relevant documents (open-book), then an LLM generates the answer (closed-book style generation).

## 1.2 In-Context Learning

The model adapts based on examples or instructions placed directly in the prompt — *no gradient updates*.

```
Zero-Shot  →  One-Shot  →  Few-Shot
(cheapest,      (1 example)   (2-8 examples,
 lowest accuracy)              highest accuracy)
```

> Whether examples help depends on the model being large enough to use them. Watch what a 125M-parameter model does with these prompts.

In [ ]:
prompt_zero = "Translate the following English sentence to French: 'Hello, how are you?'"

result = pipeline_model(prompt_zero)
print("Zero-Shot output:")
print(result[0]["generated_text"])

[transformers] Both `max_new_tokens` (=50) and `max_length`(=21) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Zero-Shot output:
Translate the following English sentence to French: 'Hello, how are you?'  I don't like saying "Hello" at all.
I don't like saying "Hello" at all either.
I don't like saying "Hello" at all either.


In [ ]:
prompt_one = (
    "Translate the following English sentence to French: 'Good morning!'\n\n"
    "Example: 'Hello, how are you?' -> 'Bonjour, comment ça va ?'"
)

result = pipeline_model(prompt_one)
print("One-Shot output:")
print(result[0]["generated_text"])

[transformers] Both `max_new_tokens` (=50) and `max_length`(=21) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


One-Shot output:
Translate the following English sentence to French: 'Good morning!'

Example: 'Hello, how are you?' -> 'Bonjour, comment ça va ?'

Example: 'Good morning, how are you?' -> 'Bonjour, comment ça va ?'

Example: 'Good morning, how are you?' -> 'Bonjour, comment ça va?


In [ ]:
prompt_few = (
    "Translate the following English sentences to French:\n\n"
    "Input: 'Hello, how are you?' -> 'Bonjour, comment ça va ?'\n"
    "Input: 'Good morning!' -> 'Bonjour!'\n\n"
    "Input: 'See you later!' ->"
)

result = pipeline_model(prompt_few)
print("Few-Shot output:")
print(result[0]["generated_text"])

[transformers] Both `max_new_tokens` (=50) and `max_length`(=21) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Few-Shot output:
Translate the following English sentences to French:

Input: 'Hello, how are you?' -> 'Bonjour, comment ça va ?'
Input: 'Good morning!' -> 'Bonjour!'

Input: 'See you later!' -> 'Bonjour, comment ça va ?'
Input: 'Hi, I'm sorry if I offended you. I just wanted to leave you with a message of help.'
Input: 'Thank you, you're welcome' ->


### What just happened?

`opt-125m` mostly copies or garbles the prompt: zero-, one-, and few-shot all fail. That's expected.

- It has **125M parameters** and was **never instruction-tuned**, so "Translate…" is just more text to continue, not a task to perform.
- In-context learning and instruction following are **emergent abilities** that appear at much larger scale (lecture slides 74 and 77).

In section **2.0** you'll send the same three prompts to a large model and compare.

### Aside: zero-shot classification without a generative model

This is a different trick from in-context learning. `facebook/bart-large-mnli` is a natural-language-inference model: for each candidate label it scores whether the text entails *"This example is {label}."* and ranks the labels. It cannot use few-shot examples, so we pass only the text to classify.

*Downloads ~1.6 GB; skip this cell if you're short on time.*

In [ ]:
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

result = classifier("The service was excellent.", candidate_labels=["positive", "negative"])
print(f"Label: {result['labels'][0]}  (score {result['scores'][0]:.2f})")

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Label: positive  (score 1.00)


### In-Context Learning — Strengths & Weaknesses

| Approach | Strengths | Weaknesses |
|----------|-----------|------------|
| **Zero-Shot** | No examples needed; fast | Less accurate on complex/niche tasks |
| **One-Shot** | Clarifies format cheaply | Single example may not generalise |
| **Few-Shot** | Better generalisation; more robust | Longer prompts; higher token cost |

> **Rule of thumb:** Start with zero-shot. If accuracy is insufficient, add 2–5 examples.

## 1.3 Fine-Tuning *(optional: covered in Weeks 5–6, you can skip to Part 2)*

Fine-tuning continues training a pre-trained model on a smaller, task-specific dataset — it *updates the model's weights*.

```
Pre-Training                        Fine-Tuning
────────────                        ───────────
Large general corpus                Small task-specific dataset
→ Broad language understanding      → Task specialisation
→ Expensive (months, $millions)     → Cheap (hours to days)
```

### Common Fine-Tuning Use Cases

| Task | Description |
|------|-------------|
| **Text-to-SQL** | Convert natural language queries into SQL |
| **Sentiment Analysis** | Classify text as positive/negative/neutral |
| **Named Entity Recognition** | Extract names, dates, organisations |
| **Summarisation** | Condense long documents |
| **Code generation** | Generate or complete code from natural language |

## 1.4 Instruction Fine-Tuning *(optional)*

Trained on a *diverse collection of tasks*, each framed as an explicit instruction.
Goal is *generality*: learn to **follow instructions** across many different tasks.

### Alpaca Prompt Format (you'll use it in Week 6)

```
<s>### Instruction:
{instruction}

### Input:
{input}

### Response:
{response}</s>
```

---
# Part 2 — Prompt Engineering with OpenAI & LangChain

- Basic prompting (system + user roles)
- Zero-shot and few-shot with LangChain's `FewShotPromptTemplate`
- Role prompting and chain prompting

> **API change:** openai v1.0+ uses `openai.OpenAI()` client. The old `openai.ChatCompletion.create()` is gone.

In [ ]:
from openai import OpenAI

# FIX: openai v1.0+ uses an explicit client object.
# The client reads OPENAI_API_KEY from the environment automatically.
client = OpenAI()

def chat(system: str, user: str, model: str = MODEL) -> str:
    """Thin wrapper around the Chat Completions API."""
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": user},
        ]
    )
    # FIX: v1 returns objects, not dicts — use .message.content not ["message"]["content"]
    return response.choices[0].message.content

## 2.0 Revisit Part 1: the same prompts on a large model

The three translation prompts from section 1.2, now sent to `MODEL`. Compare with the `opt-125m` outputs: what changed, and what does that tell you about where in-context learning comes from?

In [ ]:
try:
    prompts = {"Zero-shot": prompt_zero, "One-shot": prompt_one, "Few-shot": prompt_few}
except NameError:
    raise RuntimeError("Run section 1.2 first; this cell reuses its three prompts.")

for name, p in prompts.items():
    print(f"--- {name} ---")
    print(chat("You are a helpful assistant.", p), "\n")

--- Zero-shot ---
Bonjour, comment ça va ? 

--- One-shot ---
Good morning! -> Bonjour ! 

--- Few-shot ---
'See you later!' -> 'À plus tard !' 



## 2.1 Basic Prompting

- **System message** — sets the model's role, tone, and constraints
- **User message** — the actual request

Being specific in the system message dramatically improves output quality.

In [ ]:
system = "You are a helpful assistant whose goal is to help write stories."

prompt = (
    "Continue the following story. Write no more than 50 words.\n\n"
    "Once upon a time, in a world where animals could speak, a courageous "
    "mouse named Benjamin decided to"
)

print(chat(system, prompt))

explore the vast forest beyond his village. With a tiny satchel and a heart full of dreams, he set off to find the legendary Golden Acorn, believed to grant wisdom to any creature who possessed it. His adventure was about to begin.


In [ ]:
system = "You are a skilled copywriter whose goal is to write captivating product descriptions."

prompt = (
    "Write a captivating product description for a luxurious, handcrafted, "
    "limited-edition fountain pen made from rosewood and gold. "
    "Write no more than 50 words."
)

print(chat(system, prompt))

Indulge in elegance with our handcrafted limited-edition fountain pen, merging rich rosewood and radiant gold. Each piece is a masterpiece, designed for those who write their legacy in luxury. Elevate every word—own a symbol of timeless craftsmanship and sophistication.


## 2.2 Zero-Shot Prompting

Describe the task without examples. The model relies entirely on its pre-trained knowledge.

In [ ]:
system = "You are a helpful assistant whose goal is to write short poems."
prompt_template = "Write a short poem about {topic}."

print(chat(system, prompt_template.format(topic="summer")))

Golden rays and skies so blue,  
Summer’s warmth comes shining through.  
Whispering winds and nights so bright,  
Days of joy and pure delight.


## 2.3 In-Context Learning via Chat History

Inject examples by interleaving `user` / `assistant` turns in the `messages` list.
This is the chat-API equivalent of few-shot prompting.

In [ ]:
system = "You are a helpful assistant whose goal is to write short poems."
prompt_template = "Write a short poem about {topic}."

examples = {
    "nature": "Birdsong fills the air,\nMountains high and valleys deep,\nNature's music sweet.",
    "winter": "Snow blankets the ground,\nSilence is the only sound,\nWinter's beauty found.",
}

# FIX: use client.chat.completions.create (openai v1.0+)
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system",    "content": system},
        {"role": "user",      "content": prompt_template.format(topic="nature")},
        {"role": "assistant", "content": examples["nature"]},
        {"role": "user",      "content": prompt_template.format(topic="winter")},
        {"role": "assistant", "content": examples["winter"]},
        {"role": "user",      "content": prompt_template.format(topic="summer")},
    ]
)
# FIX: attribute access, not dict access
print(response.choices[0].message.content)

Golden sun above,  
Warmth that wraps the earth in love,  
Summer skies so bright.


## 2.4 Few-Shot Prompting with LangChain

`FewShotPromptTemplate` makes it easy to define a reusable example schema
and automatically format/inject examples into the prompt.

> **Import change (LangChain 0.3+):** `from langchain_core.prompts import ...` and `from langchain_openai import ChatOpenAI`

In [ ]:
from langchain_core.prompts import PromptTemplate, FewShotPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=MODEL, temperature=0)

examples = [
    {"color": "red",   "emotion": "passion"},
    {"color": "blue",  "emotion": "serenity"},
    {"color": "green", "emotion": "tranquility"},
]

example_prompt = PromptTemplate(
    input_variables=["color", "emotion"],
    template="Color: {color}\nEmotion: {emotion}\n",
)

few_shot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix="Here are some examples of colors and the emotions associated with them:\n",
    suffix="\nNow, given a new color, identify the emotion associated with it:\n\nColor: {input}\nEmotion:",
    input_variables=["input"],
    example_separator="\n",
)

# LCEL: prompt | llm | output parser  (replaces the deprecated LLMChain)
chain = few_shot_prompt | llm | StrOutputParser()
response = chain.invoke({"input": "purple"})
print("Color: purple")
print("Emotion:", response)

Color: purple
Emotion: Color: purple  
Emotion: creativity


In [ ]:
examples = [
    {
        "query": "What's the secret to happiness?",
        "answer": "Finding balance in life and learning to enjoy the small moments.",
    },
    {
        "query": "How can I become more productive?",
        "answer": "Try prioritising tasks, setting goals, and maintaining a healthy work-life balance.",
    },
]

example_prompt = PromptTemplate(
    input_variables=["query", "answer"],
    template="User: {query}\nAI: {answer}\n",
)

few_shot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix=(
        "The following are excerpts from conversations with an AI life coach. "
        "The assistant provides insightful and practical advice.\n"
    ),
    suffix="\nUser: {query}\nAI:",
    input_variables=["query"],
    example_separator="\n\n",
)

chain = few_shot_prompt | llm | StrOutputParser()
user_query = "What are some tips for improving communication skills?"
response = chain.invoke({"query": user_query})
print("User:", user_query)
print("AI:", response)

User: What are some tips for improving communication skills?
AI: Improving communication skills involves active listening, being clear and concise, and practicing empathy. Additionally, paying attention to non-verbal cues and asking open-ended questions can help foster better understanding in conversations. Regularly seeking feedback and engaging in conversations with diverse groups can also enhance your communication abilities.


## 2.5 Role Prompting

Assigning the model a vivid, specific **persona** guides tone, vocabulary, and reasoning style.

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=MODEL, temperature=0)

template = """As a futuristic robot band conductor, help me come up with a song title.
What's a cool song title for a song about {theme} in the year {year}?"""

prompt = PromptTemplate(input_variables=["theme", "year"], template=template)
chain = prompt | llm | StrOutputParser()

response = chain.invoke({"theme": "interstellar travel", "year": "3030"})
print("Theme: interstellar travel | Year: 3030")
print("Title:", response)

Theme: interstellar travel | Year: 3030
Title: How about **"Quantum Horizons 3030"**?  
It captures the futuristic vibe, the vastness of space, and the cutting-edge technology of interstellar travel!


## 2.6 Chain Prompting

Connect multiple prompts so the output of one becomes the input to the next.

```
Prompt 1: "Who invented X?"  →  answer: "Person A"
                                          ↓
Prompt 2: "Describe Person A's theory of X."
```

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=MODEL, temperature=0)

# Step 1: identify the scientist
chain_1 = (
    PromptTemplate(
        template="What is the name of the famous scientist who developed the theory of general relativity?\nAnswer: ",
        input_variables=[],
    )
    | llm | StrOutputParser()
)
scientist = chain_1.invoke({}).strip()
print("Step 1 — Scientist:", scientist)

# Step 2: use that answer as input to the next chain
chain_2 = (
    PromptTemplate(
        template="Provide a brief description of {scientist}'s theory of general relativity.\nAnswer: ",
        input_variables=["scientist"],
    )
    | llm | StrOutputParser()
)
fact = chain_2.invoke({"scientist": scientist})
print("\nStep 2 — Description:")
print(fact)

Step 1 — Scientist: Answer: Albert Einstein

Step 2 — Description:
Albert Einstein's theory of general relativity is a fundamental theory of gravitation that describes gravity not as a force, but as a curvature of spacetime caused by the presence of mass and energy. It explains how massive objects like stars and planets warp the fabric of spacetime, influencing the motion of other objects and the path of light. This theory has been confirmed by numerous experiments and observations and is essential for understanding phenomena such as black holes, gravitational waves, and the expansion of the universe.


## 2.7 Bad Prompt Practices vs. Good Practices

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=MODEL, temperature=0)

# BAD: vague — no format, depth, tone, or audience specified
bad_chain = PromptTemplate(input_variables=["topic"], template="Tell me something about {topic}.") | llm | StrOutputParser()
print("=== BAD prompt output ===")
print(bad_chain.invoke({"topic": "dogs"}))

# GOOD: constrained — role, length, and focus all specified
good_chain = PromptTemplate(
    input_variables=["scientist"],
    template=(
        "In 2–3 sentences, explain {scientist}'s most important scientific contribution "
        "and why it changed our understanding of the universe.\nAnswer: "
    ),
) | llm | StrOutputParser()
print("\n=== GOOD prompt output ===")
print(good_chain.invoke({"scientist": "Albert Einstein"}))

=== BAD prompt output ===
Sure! Dogs are domesticated mammals and are one of the most popular pets worldwide. They belong to the species *Canis lupus familiaris* and are descendants of wolves. Dogs have been bred for thousands of years for various purposes, including hunting, herding, guarding, and companionship. They are known for their loyalty, intelligence, and strong bond with humans. Different breeds vary widely in size, appearance, and temperament, from tiny Chihuahuas to large Great Danes. Dogs also have an excellent sense of smell and hearing, which makes them valuable in roles such as search and rescue, detection, and therapy.

=== GOOD prompt output ===
Albert Einstein's most important scientific contribution is the theory of relativity, which includes the special and general theories. This work fundamentally changed our understanding of space, time, and gravity by showing that they are interconnected and relative rather than absolute, revolutionizing physics and leading to n

### Tips for Effective Prompting

| Tip | Why it matters |
|-----|---------------|
| **Be specific** | Vague instructions produce vague output |
| **Set a persona** | Roles guide tone, vocabulary, and depth |
| **Specify format** | "In 2–3 sentences", "as a JSON list", "no more than 50 words" |
| **Provide examples** | Few-shot examples reduce ambiguity dramatically |
| **Constrain chain outputs** | Each step should produce a predictable, parseable value |
| **Iterate** | Prompting is empirical — test and refine |

---
# Part 3 — Text Splitters

| Splitter | Splits on | Best for |
|----------|-----------|----------|
| `CharacterTextSplitter` | A single character (default `\n\n`) | Simple, fast chunking |
| `RecursiveCharacterTextSplitter` | Hierarchy of separators | General-purpose; most recommended |
| `NLTKTextSplitter` | Sentence boundaries | Preserving sentence integrity |
| `MarkdownTextSplitter` | Markdown headers/code blocks | Markdown documents |
| `TokenTextSplitter` | Token count (via tiktoken) | When you need precise token budgets |

> **Import change (LangChain 0.3+):**
> - Loaders: `from langchain_community.document_loaders import PyPDFLoader`
> - Splitters: `from langchain_text_splitters import ...`

In [ ]:
# Download a RAG survey paper from arXiv as our example long document
!wget -q -O best_rag.pdf https://arxiv.org/pdf/2407.01219

# FIX: import from langchain_community (moved out of langchain core in 0.3+)
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("best_rag.pdf")
pages = loader.load()  # one Document per page; load_and_split() would already chunk them
print(f"PDF loaded: {len(pages)} pages")

/tmp/ipykernel_532/3421206419.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


PDF loaded: 22 pages


In [ ]:
# FIX: import from langchain_text_splitters (standalone package in 0.3+)
from langchain_text_splitters import CharacterTextSplitter

splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=20)
chunks = splitter.split_documents(pages)

print(f"Total chunks: {len(chunks)}")
print(f"Longest chunk: {max(len(c.page_content) for c in chunks)} characters (target: 1000)")
print("\nFirst chunk preview:")
print(chunks[0].page_content[:300])

Total chunks: 22
Longest chunk: 4679 characters (target: 1000)

First chunk preview:
Searching for Best Practices in Retrieval-Augmented
Generation
Xiaohua Wang, Zhenghua Wang, Xuan Gao, Feiran Zhang,
Yixin Wu, Zhibo Xu, Tianyuan Shi, Zhengyuan Wang, Shizheng Li,
Qi Qian, Ruicheng Yin, Changze Lv, Xiaoqing Zheng∗, Xuanjing Huang
School of Computer Science, Fudan University, Shanghai


**Look at the longest chunk.** `CharacterTextSplitter` splits only on its single separator (`"\n\n"` by default). Text extracted from PDFs has few blank lines, so chunks can come out much larger than `chunk_size`. `RecursiveCharacterTextSplitter` falls back to `"\n"`, then `" "`, then `""` until every chunk fits:

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

rec_chunks = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100).split_documents(pages)
print(f"Recursive: {len(rec_chunks)} chunks | longest {max(len(c.page_content) for c in rec_chunks)} characters")

Recursive: 98 chunks | longest 999 characters


In [ ]:
sample_text = (
    "Hello, my name is Utku.\n"
    "I am learning about LLMs and text processing.\n\n"
    "Text splitters are a crucial component in RAG pipelines.\n"
    "They divide documents into chunks that fit within the model context window."
)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=80,
    chunk_overlap=15,
    length_function=len,
)
chunks = splitter.create_documents([sample_text])

print(f"Number of chunks: {len(chunks)}")
for i, chunk in enumerate(chunks):
    print(f"\nChunk {i+1}: {repr(chunk.page_content)}")

Number of chunks: 3

Chunk 1: 'Hello, my name is Utku.\nI am learning about LLMs and text processing.'

Chunk 2: 'Text splitters are a crucial component in RAG pipelines.'

Chunk 3: 'They divide documents into chunks that fit within the model context window.'


In [ ]:
import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

from langchain_text_splitters import NLTKTextSplitter

splitter = NLTKTextSplitter(chunk_size=200, chunk_overlap=0)
chunks = splitter.split_text(sample_text)

print(f"Number of chunks: {len(chunks)}")
for i, chunk in enumerate(chunks):
    print(f"\nChunk {i+1}: {repr(chunk)}")

Number of chunks: 2

Chunk 1: 'Hello, my name is Utku.\n\nI am learning about LLMs and text processing.\n\nText splitters are a crucial component in RAG pipelines.'

Chunk 2: 'They divide documents into chunks that fit within the model context window.'


In [ ]:
from langchain_text_splitters import MarkdownTextSplitter

markdown_text = """
# Welcome to My Blog!

## Introduction
Hello everyone! I specialise in Python, Java, and JavaScript.

My favourite languages:
1. Python
2. JavaScript
3. Java

## About this Blog
I will share tutorials and thoughts on the latest technology trends.

```python
def say_hello(name):
    print(f"Hello, {name}!")
```

## Contact Me
Feel free to reach out on Twitter or by email.
"""

splitter = MarkdownTextSplitter(chunk_size=100, chunk_overlap=0)
chunks = splitter.create_documents([markdown_text])

print(f"Number of chunks: {len(chunks)}")
for i, chunk in enumerate(chunks):
    print(f"\nChunk {i+1}:")
    print(chunk.page_content)

Number of chunks: 6

Chunk 1:
# Welcome to My Blog!

Chunk 2:
## Introduction
Hello everyone! I specialise in Python, Java, and JavaScript.

Chunk 3:
My favourite languages:
1. Python
2. JavaScript
3. Java

Chunk 4:
## About this Blog
I will share tutorials and thoughts on the latest technology trends.

Chunk 5:
```python
def say_hello(name):
    print(f"Hello, {name}!")

Chunk 6:
```

## Contact Me
Feel free to reach out on Twitter or by email.


In [ ]:
from langchain_text_splitters import TokenTextSplitter

splitter = TokenTextSplitter(chunk_size=100, chunk_overlap=20)
chunks = splitter.split_text(sample_text)

print(f"Number of chunks: {len(chunks)}")
for i, chunk in enumerate(chunks):
    print(f"\nChunk {i+1}: {repr(chunk)}")

Number of chunks: 1

Chunk 1: 'Hello, my name is Utku.\nI am learning about LLMs and text processing.\n\nText splitters are a crucial component in RAG pipelines.\nThey divide documents into chunks that fit within the model context window.'


### Which splitter should I use?

```
Is the document Markdown?          → MarkdownTextSplitter
Do you need exact token budgets?   → TokenTextSplitter
Is sentence integrity critical?    → NLTKTextSplitter
Everything else (default choice):  → RecursiveCharacterTextSplitter
```

**Overlap tip:** Set `chunk_overlap` to ~10–15% of `chunk_size` to prevent
important context from being lost at chunk boundaries.

---
# Part 4 — LangChain LCEL Chains

**LCEL (LangChain Expression Language)** is the current LangChain pattern — chains are built by piping runnables together with `|`:

```python
chain = prompt | llm | output_parser
result = chain.invoke({"key": "value"})   # returns a string (or list, dict, etc.)
```

This section covers:
1. **Basic LCEL chain** — prompt → LLM → string
2. **Conversation with memory** — multi-turn dialogue via explicit message history
3. **Composing two chains** — combine outputs of parallel chains

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=MODEL, temperature=0)

## 4.1 Basic LCEL Chain

`prompt | llm | StrOutputParser()` is the LCEL equivalent of the old `LLMChain`.
`.invoke({...})` runs the full pipeline and returns a plain string.

In [ ]:
prompt = PromptTemplate.from_template("What is a word to replace the following: {word}?")
chain = prompt | llm | StrOutputParser()

print("Result:", chain.invoke({"word": "artificial"}))

Result: Here are some synonyms for **artificial** depending on the context:

- **Synthetic**
- **Man-made**
- **Fake**
- **Imitation**
- **Manufactured**
- **Fabricated**
- **Constructed**
- **Simulated**

If you provide the specific sentence or context, I can suggest the most suitable replacement!


In [ ]:
prompt = PromptTemplate(
    template="Looking at the context of '{context}'. What is an appropriate word to replace the following: {word}?",
    input_variables=["word", "context"],
)
chain = prompt | llm | StrOutputParser()

print("Context=object:", chain.invoke({"word": "fan", "context": "object"}))
print("Context=humans:", chain.invoke({"word": "fan", "context": "humans"}))

Context=object: To suggest an appropriate replacement for the word **"fan"** in the context of **"object"**, it would help to know the specific context or sentence where "fan" is used. However, generally speaking:

- If **"fan"** refers to a **device that moves air**, appropriate synonyms or replacements could be:
  - **blower**
  - **ventilator**
  - **air circulator**
  - **cooler** (if it cools air)
  - **propeller** (in some contexts)

- If **"fan"** refers to a **person who admires or supports something**, then it’s not an object, so replacement wouldn’t apply in the "object" context.

- If **"fan"** refers to a **folding hand fan** (the handheld object), replacements could be:
  - **hand fan**
  - **folding fan**
  - **fan blade** (if referring to parts)

If you provide the sentence or more context, I can give a more precise replacement!
Context=humans: In the context of "humans," an appropriate word to replace "fan" could be:

- **enthusiast**
- **admirer**
- **supporter**
- **d

In [ ]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

output_parser = CommaSeparatedListOutputParser()

prompt = PromptTemplate(
    template="List all possible words as substitutes for 'artificial' as a comma-separated list.",
    input_variables=[],
)

# Pipe directly into the output parser — returns a Python list, no manual parsing needed
chain = prompt | llm | output_parser
parsed = chain.invoke({})
print("Parsed list:", parsed)

Parsed list: ['synthetic', 'man-made', 'manufactured', 'fabricated', 'imitation', 'faux', 'ersatz', 'simulated', 'contrived', 'engineered', 'processed', 'constructed', 'fake', 'false', 'replica', 'mock', 'pseudo', 'counterfeit', 'forged', 'created']


## 4.2 Conversation with Memory

The modern way to handle multi-turn conversation is a plain **message list** —
`SystemMessage`, `HumanMessage`, and `AIMessage` objects appended on each turn.
The full list is sent to the LLM every time, making memory completely transparent.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.output_parsers import CommaSeparatedListOutputParser

output_parser = CommaSeparatedListOutputParser()

system_msg = SystemMessage(content="List possible word substitutes for 'artificial' as a comma-separated list.")
history = []

def chat_with_memory(user_input: str) -> str:
    history.append(HumanMessage(content=user_input))
    response = llm.invoke([system_msg] + history)
    history.append(AIMessage(content=response.content))
    return response.content

turn1_raw = chat_with_memory("Give me the first 3 options.")
print("Turn 1:", output_parser.parse(turn1_raw))

# The model remembers what it already listed because history is resent each turn
turn2_raw = chat_with_memory("And the next 4?")
print("Turn 2:", output_parser.parse(turn2_raw))

Turn 1: ['synthetic', 'fake', 'imitation']
Turn 2: ['man-made', 'manufactured', 'simulated', 'ersatz']


In [ ]:
# verbose equivalent: print the full message list sent on each turn
# This replaces the old verbose=True on ConversationChain
history2 = []

def chat_with_memory_verbose(user_input: str) -> str:
    history2.append(HumanMessage(content=user_input))
    messages = [system_msg] + history2
    print("--- Full context sent to LLM ---")
    for m in messages:
        print(f"  [{type(m).__name__}] {m.content[:80]}")
    print()
    response = llm.invoke(messages)
    history2.append(AIMessage(content=response.content))
    return response.content

raw = chat_with_memory_verbose("Give me the first 3 options.")
print("Parsed:", output_parser.parse(raw))

--- Full context sent to LLM ---
  [SystemMessage] List possible word substitutes for 'artificial' as a comma-separated list.
  [HumanMessage] Give me the first 3 options.

Parsed: ['synthetic', 'fake', 'imitation']


## 4.3 Composing Two LCEL Chains

Run two chains on the same input and combine their outputs —
the LCEL equivalent of the old `ConcatenateChain` subclass.

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=MODEL, temperature=0)

chain_meaning = (
    PromptTemplate(input_variables=["word"], template="In one sentence, what does '{word}' mean?")
    | llm | StrOutputParser()
)
chain_synonym = (
    PromptTemplate(input_variables=["word"], template="Give one single-word synonym for '{word}'.")
    | llm | StrOutputParser()
)

def concatenate_chains(word: str) -> str:
    """Run both chains on the same input and join their outputs."""
    return chain_meaning.invoke({"word": word}) + "\n---\n" + chain_synonym.invoke({"word": word})

result = concatenate_chains("artificial")
print("Concatenated output:")
print(result)

Concatenated output:
"Artificial" means made or produced by human beings rather than occurring naturally.
---
Synthetic


---
# Conclusion

| Part | Key Takeaway |
|------|-------------|
| **1 — LLM Fundamentals** | Open-book QA extracts answer spans (start/end logits); closed-book generates from training memory; in-context learning trades accuracy vs. prompt cost |
| **2 — Prompt Engineering** | Specificity, format constraints, and few-shot examples are the three most impactful levers |
| **3 — Text Splitters** | `RecursiveCharacterTextSplitter` is the safe default; use `TokenTextSplitter` for precise token budgets |
| **4 — LangChain Chains** | Chains compose prompts, memory, and parsers; printing the full message list (section 4.2) is your best debugging tool |

### Next steps

- **RAG (Week 2):** Combine text splitters (Part 3) with vector databases (FAISS, Chroma) and an LLM chain (Part 4)
- **Fine-tuning (Week 6):** Mistral-7B + QLoRA + SFTTrainer on Alpaca (requires a GPU runtime)
- **RLHF (Week 9):** Go beyond instruction fine-tuning with reward models and PPO/DPO